# Task 2: Comparación entre modelo estándar y modelo destilado en CPU/CUDA

### Integrantes

* Sergio Orellana 221122
* Rodrigo Mansilla 22611
* Ricardo Chuy 221007


## Prompt utilizado con IA y justificación

**Prompt utilizado:**

> Actúa como ingeniero de visión por computadora especializado en modelos de difusión con Hugging Face Diffusers. Genera un notebook de Python que compare Stable Diffusion v1.5 a 50 pasos contra un modelo destilado Turbo a 4 pasos, usando la misma semilla, el mismo prompt, medición de tiempo con `time.time()` y medición de VRAM con `torch.cuda.max_memory_allocated()`. Incluye visualización lado a lado, tabla de métricas y una discusión técnica sobre destilación, U-Net, VAE, scheduler y despliegue en producción.

**Por qué funcionó:**

El prompt funcionó porque especifica explícitamente los dos escenarios, las métricas obligatorias y los componentes técnicos que exige el laboratorio. Además, delimita el contexto del análisis: no solo pide generar imágenes, sino comparar el flujo completo `Tensor latente -> U-Net -> Scheduler -> VAE -> Píxeles` para tomar una decisión arquitectónica.


## Instalación de dependencias


In [1]:
# Para Colab:
# !pip install -q -U diffusers transformers accelerate safetensors pillow matplotlib pandas ipywidgets


## Imports y configuración del entorno


In [2]:
%matplotlib inline

import gc
import os
import time
import threading

import psutil
import torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

from diffusers import StableDiffusionPipeline, AutoPipelineForText2Image


[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.


In [3]:
PREFER_CUDA = False
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = "cuda" if PREFER_CUDA and CUDA_AVAILABLE else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
SEED = 42

if PREFER_CUDA and not CUDA_AVAILABLE:
    print("PREFER_CUDA=True, pero PyTorch no detectó CUDA. Se usará CPU.")

PROMPT = "A highly detailed cinematic and futuristic fruit glowing in a cyberpunk laboratory, neon lights, 4k resolution"

HEIGHT = 512
WIDTH = 512

NUM_THREADS = min(os.cpu_count() or 1, 8)
torch.set_num_threads(NUM_THREADS)

os.makedirs("outputs_task2", exist_ok=True)

print("Dispositivo usado:", DEVICE)
print("Precisión usada:", DTYPE)
print("CUDA disponible:", CUDA_AVAILABLE)
print("Hilos de CPU usados por PyTorch:", torch.get_num_threads())
if DEVICE == "cuda":
    print("GPU usada:", torch.cuda.get_device_name(0))
print("Prompt:", PROMPT)
print("Seed:", SEED)


Dispositivo usado: cpu
Precisión usada: torch.float32
CUDA disponible: False
Hilos de CPU usados por PyTorch: 8
Prompt: A highly detailed cinematic and futuristic fruit glowing in a cyberpunk laboratory, neon lights, 4k resolution
Seed: 42


## Funciones auxiliares para medir rendimiento en CPU o CUDA


In [4]:
class PeakRAMMonitor:
    def __init__(self, interval=0.05):
        self.interval = interval
        self.process = psutil.Process(os.getpid())
        self.peak_bytes = 0
        self._running = False
        self._thread = None

    def _watch(self):
        while self._running:
            rss = self.process.memory_info().rss
            if rss > self.peak_bytes:
                self.peak_bytes = rss
            time.sleep(self.interval)

    def start(self):
        self.peak_bytes = self.process.memory_info().rss
        self._running = True
        self._thread = threading.Thread(target=self._watch, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread is not None:
            self._thread.join()
        self.peak_bytes = max(self.peak_bytes, self.process.memory_info().rss)

    @property
    def peak_mb(self):
        return self.peak_bytes / (1024 ** 2)

    @property
    def peak_gb(self):
        return self.peak_bytes / (1024 ** 3)


def using_cuda():
    return DEVICE == "cuda" and torch.cuda.is_available()


def limpiar_memoria():
    gc.collect()
    if using_cuda():
        torch.cuda.empty_cache()


def optimizar_pipeline(pipe):
    pipe = pipe.to(DEVICE)
    if hasattr(pipe, "enable_attention_slicing"):
        pipe.enable_attention_slicing()
    if hasattr(pipe, "enable_vae_slicing"):
        pipe.enable_vae_slicing()
    return pipe


def generar_y_medir(
    pipe,
    model_label,
    model_id,
    steps,
    output_path,
    guidance_scale=None,
):
    limpiar_memoria()

    generator = torch.Generator(device=DEVICE).manual_seed(SEED)

    kwargs = {
        "prompt": PROMPT,
        "num_inference_steps": steps,
        "generator": generator,
        "height": HEIGHT,
        "width": WIDTH,
    }

    if guidance_scale is not None:
        kwargs["guidance_scale"] = guidance_scale

    if using_cuda():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    monitor = PeakRAMMonitor()
    monitor.start()
    inicio = time.time()

    with torch.inference_mode():
        image = pipe(**kwargs).images[0]

    if using_cuda():
        torch.cuda.synchronize()

    fin = time.time()
    monitor.stop()

    tiempo_segundos = fin - inicio
    ram_mb = monitor.peak_mb
    ram_gb = monitor.peak_gb

    if using_cuda():
        vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    else:
        vram_mb = "N/A (CPU)"
        vram_gb = "N/A (CPU)"

    image.save(output_path)

    metricas = {
        "Modelo usado": model_label,
        "ID del modelo": model_id,
        "Dispositivo": DEVICE,
        "Precisión": str(DTYPE).replace("torch.", ""),
        "Pasos": steps,
        "Tiempo de Ejecución (s)": round(tiempo_segundos, 3),
        "VRAM (MB)": round(vram_mb, 2) if isinstance(vram_mb, float) else vram_mb,
        "VRAM (GB)": round(vram_gb, 3) if isinstance(vram_gb, float) else vram_gb,
        "RAM pico proceso (MB)": round(ram_mb, 2),
        "RAM pico proceso (GB)": round(ram_gb, 3),
        "Imagen guardada": output_path,
    }

    return image, metricas


# Escenario A: Modelo estándar - Costoso


In [ ]:
standard_model_id = "sd-legacy/stable-diffusion-v1-5"

pipe_a = StableDiffusionPipeline.from_pretrained(
    standard_model_id,
    torch_dtype=DTYPE,
    safety_checker=None,
    feature_extractor=None,
    requires_safety_checker=False,
)

pipe_a = optimizar_pipeline(pipe_a)

image_a, metrics_a = generar_y_medir(
    pipe=pipe_a,
    model_label="Escenario A - Stable Diffusion v1.5",
    model_id=standard_model_id,
    steps=50,
    output_path="outputs_task2/escenario_A_sd15_50_steps.png",
    guidance_scale=7.5,
)

display(image_a)
metrics_a


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

## Liberación de memoria entre modelos

Después de ejecutar el Escenario A, se elimina el pipeline de memoria. De este modo, la medición del Escenario B no queda contaminada por tensores o pesos del modelo anterior.


In [ ]:
del pipe_a
limpiar_memoria()
print("Memoria RAM liberada antes de cargar el modelo destilado.")


# Escenario B: Modelo destilado - Eficiente

A continuación, se carga `stabilityai/sd-turbo`, un modelo Turbo diseñado para generar imágenes coherentes en muy pocos pasos. En este caso se usan 4 pasos de inferencia para simular el paradigma de un modelo eficiente similar a lo descrito en el laboratorio.

Para SD-Turbo se usa `guidance_scale=0.0`, ya que este tipo de modelo Turbo no depende de classifier-free guidance de la misma manera que el modelo estándar. En CPU el modelo se carga en `torch.float32`; si se activa CUDA, se cargará en `torch.float16` para reducir consumo de VRAM.


In [ ]:
turbo_model_id = "stabilityai/sd-turbo"

pipe_b = AutoPipelineForText2Image.from_pretrained(
    turbo_model_id,
    torch_dtype=DTYPE,
    safety_checker=None,
    feature_extractor=None,
    requires_safety_checker=False,
)

pipe_b = optimizar_pipeline(pipe_b)

image_b, metrics_b = generar_y_medir(
    pipe=pipe_b,
    model_label="Escenario B - SD-Turbo",
    model_id=turbo_model_id,
    steps=4,
    output_path="outputs_task2/escenario_B_sd_turbo_4_steps.png",
    guidance_scale=0.0,
)

display(image_b)
metrics_b


## Comparación visual lado a lado


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(image_a)
axes[0].set_title("Escenario A\nSD v1.5 - 50 pasos")
axes[0].axis("off")

axes[1].imshow(image_b)
axes[1].set_title("Escenario B\nSD-Turbo - 4 pasos")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("outputs_task2/comparacion_lado_a_lado.png", dpi=150, bbox_inches="tight")
plt.show()

print("Imagen comparativa guardada en: outputs_task2/comparacion_lado_a_lado.png")


## Tabla comparativa de métricas


In [ ]:
df_metricas = pd.DataFrame([metrics_a, metrics_b])
display(df_metricas)

df_metricas.to_csv("outputs_task2/tabla_metricas_task2.csv", index=False)
print("Tabla guardada en: outputs_task2/tabla_metricas_task2.csv")


## Discusión técnica: destilación y pocos pasos

En primer lugar, el modelo estándar de difusión aprende a remover ruido gradualmente a lo largo de una trayectoria de denoising relativamente densa. Por esta razón, cuando Stable Diffusion v1.5 se ejecuta con 50 pasos, el scheduler puede actualizar el tensor latente muchas veces y la U-Net tiene suficientes oportunidades para corregir errores globales y locales antes de que el VAE convierta el latente final a píxeles.

Sin embargo, si ese mismo modelo estándar se redujera directamente a 4 pasos, el proceso quedaría submuestreado. Es decir, el scheduler saltaría demasiadas posiciones de la cadena de denoising y la U-Net no tendría suficientes iteraciones para transformar el ruido inicial en una estructura visual estable. En consecuencia, el resultado tendería a conservar ruido residual, composición débil, bordes inestables o baja coherencia semántica.

En cambio, el modelo destilado del Escenario B fue entrenado para aproximar en pocos pasos el comportamiento de un modelo maestro más costoso. En términos prácticos, la destilación comprime la trayectoria de denoising: el estudiante aprende a realizar en 1 a 4 evaluaciones de red una transformación que normalmente requeriría muchas más actualizaciones del latente. Por tanto, el modelo B puede producir una imagen coherente en 4 pasos porque su U-Net y su scheduler fueron optimizados explícitamente para el régimen de pocos pasos.

Desde el flujo de datos, ambos escenarios siguen la misma estructura conceptual: primero se inicializa un tensor latente de ruido; después, la U-Net predice componentes de ruido condicionada por el prompt y por el timestep; luego, el scheduler actualiza el latente; finalmente, el VAE decodifica el latente hacia píxeles. La diferencia principal es que el modelo destilado aprendió una trayectoria más corta, mientras que el modelo estándar depende de una trayectoria más larga para llegar a una imagen estable.


## Dictamen arquitectónico para producción

Como arquitecto de IA, elegiría el **Escenario B - Modelo destilado** para una API de generación de imágenes con millones de usuarios, siempre que la calidad visual obtenida sea suficiente para el producto final. La razón principal es que el número de pasos baja de 50 a 4; por consiguiente, disminuyen el tiempo de inferencia, la ocupación del hardware por solicitud y el costo operativo por imagen.

Además, en sistemas de alta concurrencia, la latencia y el throughput son tan importantes como la fidelidad visual. Un modelo estándar de 50 pasos puede producir imágenes más refinadas, pero su costo por petición dificulta escalar el servicio sin multiplicar la cantidad de servidores. En cambio, un modelo Turbo permite responder más rápido, atender más usuarios por unidad de hardware y reducir costos de infraestructura.

No obstante, ejecutar estos modelos en CPU no sería la decisión ideal para producción real, porque la latencia puede ser demasiado alta. En definitiva, para una API masiva e interactiva escogería el modelo destilado ejecutado sobre aceleradores adecuados; para generación offline de alta fidelidad, mantendría el modelo estándar como opción premium o modo de renderizado lento.


## Discusión automática basada en las métricas medidas


In [ ]:
tiempo_a = df_metricas.loc[0, "Tiempo de Ejecución (s)"]
tiempo_b = df_metricas.loc[1, "Tiempo de Ejecución (s)"]
ram_a = df_metricas.loc[0, "RAM pico proceso (GB)"]
ram_b = df_metricas.loc[1, "RAM pico proceso (GB)"]
vram_a = df_metricas.loc[0, "VRAM (GB)"]
vram_b = df_metricas.loc[1, "VRAM (GB)"]

aceleracion = tiempo_a / tiempo_b if tiempo_b > 0 else float("inf")
reduccion_tiempo = (1 - tiempo_b / tiempo_a) * 100 if tiempo_a > 0 else 0
reduccion_ram = (1 - ram_b / ram_a) * 100 if ram_a > 0 else 0

if using_cuda() and isinstance(vram_a, (int, float)) and isinstance(vram_b, (int, float)):
    memoria_texto = (
        f"La VRAM máxima fue de **{vram_a:.3f} GB** para el Escenario A "
        f"y **{vram_b:.3f} GB** para el Escenario B."
    )
else:
    memoria_texto = (
        "Como esta ejecución se hizo en CPU, la VRAM aparece como **N/A (CPU)**. "
        "En su lugar, se reportó RAM pico del proceso."
    )

conclusion = f'''
### Conclusión cuantitativa ajustada al dispositivo

El Escenario A tardó **{tiempo_a:.3f} s** y alcanzó un pico aproximado de **{ram_a:.3f} GB** de RAM del proceso. En cambio, el Escenario B tardó **{tiempo_b:.3f} s** y alcanzó **{ram_b:.3f} GB** de RAM del proceso. Por tanto, el modelo destilado fue aproximadamente **{aceleracion:.2f}x** más rápido y redujo el tiempo de inferencia en **{reduccion_tiempo:.2f}%**.

{memoria_texto} La reducción de RAM observada fue de **{reduccion_ram:.2f}%**, aunque esta diferencia puede variar porque ambos modelos deben cargar pesos completos en memoria principal.

En conclusión, si la imagen generada por SD-Turbo mantiene una calidad visual aceptable para el usuario final, el Escenario B es la opción más razonable para despliegue masivo. No obstante, para producción real convendría ejecutarlo en GPU o aceleradores especializados, ya que CPU es útil para completar el laboratorio localmente, pero no es el entorno ideal para baja latencia.
'''

display(Markdown(conclusion))
